# Python 数据分析综合复习

课程资料从 NumPy、Pandas 基础和进阶、Matplotlib 可视化，一直到 RFM 用户分群案例，形成了一条完整的数据分析学习路径。本篇作为第 06 篇复习笔记，把这些真实模块串成一个可复用的项目流程。

> 资料目录中没有单独的第 06 个课程文件夹，因此本篇定位为课程总复习，不新增课程外的知识体系。代码均放在 Markdown 代码块中，运行前请根据实际数据文件调整路径和字段。

## 一、全课程知识地图

| 阶段 | 核心内容 | 典型问题 |
| --- | --- | --- |
| NumPy | 数组、维度、索引、运算、统计 | 如何高效处理数值数组？ |
| Pandas 基础 | Series、DataFrame、索引、筛选、排序 | 如何把表格数据读入并定位记录？ |
| Pandas 进阶 | 缺失值、合并、分组、透视表、文件读写 | 如何清洗、汇总和连接多张表？ |
| Matplotlib | 折线、柱状、直方、饼图、散点图 | 如何把分析结果表达成图？ |
| RFM 案例 | R/F/M 计算、分箱、分群、导出 | 如何把交易记录转成用户价值标签？ |

项目中通常不是线性只用一个库，而是用 NumPy 做数值基础，用 Pandas 组织表格，用 Matplotlib 或 Pyecharts 展示结果。

## 二、标准数据分析工作流

一个可复用的分析项目可以拆成八步：

1. **明确问题**：确定分析对象、时间范围、指标和最终交付形式；
2. **读取数据**：从 CSV、Excel、SQL 或 JSON 导入；
3. **认识数据**：查看 `head`、`info`、`describe`、字段类型和缺失情况；
4. **清洗数据**：统一类型、处理缺失值、重复值和异常记录；
5. **构造指标**：使用时间运算、条件表达式和分组聚合生成业务字段；
6. **探索分析**：筛选、排序、统计、合并和透视；
7. **可视化表达**：根据问题选择趋势、比较、分布或关系图；
8. **交付结果**：导出 Excel、CSV、数据库表或 HTML 图表，并记录口径。

每一步都应保留可追溯的中间结果，避免直接覆盖原始数据。

## 三、读取与检查数据

### 3.1 常见读取方式

```python
import pandas as pd

# CSV
df_csv = pd.read_csv("data/example.csv", encoding="utf-8")

# Excel
df_excel = pd.read_excel("data/example.xlsx", sheet_name="Sheet1")

# 多工作表，返回 dict[str, DataFrame]
sheets = pd.read_excel(
    "data/sales.xlsx",
    sheet_name=["2015", "2016", "2017", "2018"],
)

# SQL
# df_sql = pd.read_sql("SELECT * FROM orders", engine)
```

### 3.2 快速体检

```python
df.head()
df.tail()
df.shape
df.info()
df.describe(include="all")
df.isna().sum()
df.nunique()
```

先确认行粒度。例如订单表的一行可能是一笔订单，用户汇总表的一行则应该是一个用户；如果混淆粒度，后续聚合会重复计算。

## 四、数据类型与索引

日期、数值和分类字段应尽早转换为适合分析的类型。

```python
df["提交日期"] = pd.to_datetime(
    df["提交日期"],
    errors="coerce",
)
df["订单金额"] = pd.to_numeric(
    df["订单金额"],
    errors="coerce",
)

# 以业务键设置索引时要确认是否唯一
df = df.set_index("会员ID")
```

- `loc` 按标签选取，`iloc` 按位置选取；
- `reset_index()` 将索引恢复为列；
- `set_index()` 可以使用一列或多列建立索引；
- 参与 `merge` 的键在两边应保持相同的数据类型。

## 五、清洗数据的检查顺序

### 5.1 缺失值

```python
# 查看每列缺失数量
missing = df.isna().sum().sort_values(ascending=False)

# 删除关键字段为空的记录
cleaned = df.dropna(subset=["会员ID", "订单金额"])

# 数值字段用中位数填充的示例
cleaned["评分"] = cleaned["评分"].fillna(
    cleaned["评分"].median()
)
```

如果缺失标记是 `?` 或其他字符串，先统一成 `NaN`：

```python
cleaned = cleaned.replace("?", pd.NA)
```

### 5.2 重复与异常

```python
duplicates = df.duplicated().sum()
df = df.drop_duplicates()

# 根据业务规则删除无效金额
df = df[df["订单金额"] > 1]
```

异常值是否删除必须有业务依据。极端值可能是错误录入，也可能是真实的大额订单。

## 六、构造指标与分组聚合

### 6.1 时间特征

```python
df["year"] = df["提交日期"].dt.year
df["month"] = df["提交日期"].dt.month
df["weekday"] = df["提交日期"].dt.weekday
```

### 6.2 分组聚合

```python
user_summary = (
    df.groupby("会员ID", as_index=False)
      .agg(
          order_count=("订单号", "count"),
          total_amount=("订单金额", "sum"),
          last_order=("提交日期", "max"),
      )
)
```

命名聚合能直接生成清晰的结果列，适合后续导出。多列分组时要先确认业务口径，例如按 `year + 会员ID` 分组表示“每年每个用户”，不等于“整个周期每个用户”。

## 七、合并数据的选择

### `concat`：结构相同的表拼接

```python
all_years = pd.concat(
    [df_2015, df_2016, df_2017, df_2018],
    ignore_index=True,
)
```

### `merge`：依据键连接不同字段

```python
user_order = pd.merge(
    orders,
    members,
    how="left",
    on="会员ID",
)
```

合并前检查：

- 连接键是否为空；
- 两边键的数据类型是否一致；
- 每个键在两边是否唯一；
- 合并后行数是否符合预期。

多对多连接会造成记录数膨胀，必要时先对一侧按键聚合。

## 八、从统计结果到图表

### 8.1 按问题选图

- 时间趋势：折线图；
- 类别比较：柱状图；
- 连续变量分布：直方图；
- 少量类别占比：饼图；
- 两个数值变量关系：散点图；
- 多指标或多维关系：子图、双轴或交互式图表。

### 8.2 统一绘图模板

```python
import matplotlib.pyplot as plt
from pylab import mpl

mpl.rcParams["font.sans-serif"] = ["SimHei"]
mpl.rcParams["axes.unicode_minus"] = False

fig, ax = plt.subplots(figsize=(10, 6), dpi=100)
ax.plot(x, y, label="指标")
ax.set_title("指标变化趋势")
ax.set_xlabel("时间")
ax.set_ylabel("数值")
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend()
fig.tight_layout()
fig.savefig("result.png")
plt.show()
```

保存图像放在显示之前；标题、坐标轴、单位和图例应让图表脱离代码也能被理解。

## 九、RFM 案例的完整思路

RFM 案例是课程知识的综合应用：

1. 读取 Excel 中的多个年度订单表；
2. 删除缺失值和金额不合理的订单；
3. 合并年度表，添加年份和订单距年末的天数；
4. 按年份和会员 ID 聚合得到 R、F、M 原始值；
5. 查看分布，结合分位数与业务经验确定边界；
6. 用 `pd.cut` 对 R/F/M 分箱；
7. 注意 R 的方向与 F/M 相反；
8. 将三个标签拼接成 `rfm_group`；
9. 统计各年份分组人数并用 3D 图展示；
10. 导出 Excel，并写入 MySQL。

RFM 的关键不在于固定某一套阈值，而在于把时间口径、业务规则和分箱结果写清楚。

## 十、RFM 核心代码骨架

```python
# 读取并清洗年度表
for name in sheet_names[:-1]:
    table = sheets[name].dropna().copy()
    table = table[table["订单金额"] > 1]
    table["max_year_date"] = table["提交日期"].max()
    sheets[name] = table

# 合并、计算年份和日期间隔
orders = pd.concat(
    [sheets[name] for name in sheet_names[:-1]],
    ignore_index=True,
)
orders["year"] = orders["提交日期"].dt.year
orders["date_interval"] = (
    orders["max_year_date"] - orders["提交日期"]
).dt.days

# 用户年度 R/F/M
rfm = (
    orders.groupby(["year", "会员ID"], as_index=False)
    .agg(
        r=("date_interval", "min"),
        f=("订单号", "count"),
        m=("订单金额", "sum"),
    )
)

# 分箱打分：R 越小越好，F/M 越大越好
rfm["r_score"] = pd.cut(
    rfm["r"], [-1, 79, 255, 365],
    labels=[3, 2, 1], include_lowest=True,
)
rfm["f_score"] = pd.cut(
    rfm["f"], [0, 2, 5, 130],
    labels=[1, 2, 3], include_lowest=True,
)
rfm["m_score"] = pd.cut(
    rfm["m"], [0, 69, 1199, 206252],
    labels=[1, 2, 3], include_lowest=True,
)

rfm["rfm_group"] = (
    rfm["r_score"].astype(str)
    + rfm["f_score"].astype(str)
    + rfm["m_score"].astype(str)
)
```

## 十一、结果交付与质量校验

### 11.1 导出

```python
rfm.to_excel("sales_rfm_score.xlsx", index=False)

# 写入数据库前先创建 engine
# rfm.to_sql("sales_rfm_score", engine, index=False, if_exists="append")
```

### 11.2 交付前检查

```python
assert rfm["会员ID"].notna().all()
assert rfm["rfm_group"].notna().all()

# 每个年度的用户数
print(rfm.groupby("year")["会员ID"].nunique())

# 分组人数合计是否等于用户记录数
print(rfm["rfm_group"].value_counts(dropna=False))

# 导出文件行数
print(rfm.shape)
```

还应人工抽样检查几个用户，确认最近订单、订单次数、金额合计和分组编码符合原始记录。

## 十二、常见错误清单

### 把订单数当成用户数

订单表一行通常是一笔订单。统计用户数量要使用 `nunique()`，而不是简单 `count()`。

### 忽略 R 的评分方向

R 表示距离最近购买的间隔，天数越少越好；如果直接按数值升序打分，会把沉默用户误判为高价值用户。

### 分箱边界覆盖不全

`pd.cut` 后出现 `NaN`，通常说明最小边界、最大边界或数据类型有问题。

### 不检查合并基数

会员等级表应保证会员 ID 的连接关系符合预期，否则左连接可能造成重复行。

### 只看分组编码，不看原始指标

`333` 只是编码。运营解释必须回到 R、F、M 原始值和具体区间。

### 用固定阈值替代业务验证

课程阈值只适用于案例数据和案例业务背景，换行业、换周期或换商品后应重新评估。

## 十三、学习与项目检查表

- [ ] 能独立读取 CSV、Excel、SQL 和 JSON；
- [ ] 能用 `info`、`describe`、缺失值统计检查数据；
- [ ] 能完成日期、数值和分类字段转换；
- [ ] 能区分 `loc`、`iloc`、布尔索引和 `query`；
- [ ] 能使用 `concat` 和 `merge`，并检查合并后的行数；
- [ ] 能用 `groupby + agg` 构造业务指标；
- [ ] 能选择合适的 Matplotlib 图形并补齐图表信息；
- [ ] 能解释 `pd.cut` 的区间规则；
- [ ] 能说明 R、F、M 三个指标的方向；
- [ ] 能导出分析结果并完成行数、缺失和抽样校验。

## 本篇小结

Python 数据分析的核心能力是把业务问题转成清晰的数据口径，再用 NumPy、Pandas 和可视化工具完成从读取、清洗、计算到交付的闭环。RFM 案例展示了这一过程：技术 API 只是实现手段，时间范围、异常规则、分箱边界和结果解释才决定分析是否可靠。